# 04 — Универсальный runner контролируемых экспериментов

Этот notebook **не содержит код конкретной идеи**. Он загружает выбранный
Git-версионируемый модуль из `src/ml_project/experiments/`, пересчитывает
baseline reference и candidate на одинаковых folds, сохраняет графики,
артефакты, карточку и реестры.

| Что меняется между экспериментами | Где хранится |
|---|---|
| Выбранный модуль | `src/ml_project/experiment_config.py` |
| Гипотеза, критерии, candidate-data и candidate-models | отдельный модуль `src/ml_project/experiments/exp_xxx_*.py` |
| Общий CV, метрики и baseline | `baseline_config.BASELINE` |
| Автоматический отчёт и ручное объяснение | карточка `experiments/EXP-xxx ...md` |

> [!important]
> Не добавляйте экспериментальные ячейки в этот notebook. Для новой идеи
> создайте отдельный модуль scaffolder-командой, показанной в конце.

[Карточка проекта: как начать новый эксперимент](../README.md#как-начать-новый-эксперимент)


## 0. Перед стартом

- [ ] Baseline сохранён и понятен.
- [ ] В `experiment_config.py` выбран нужный `EXPERIMENT_MODULE`.
- [ ] В модуле нет `CHANGE ME`.
- [ ] Заранее заданы `primary_improvement_min` и `metric_guardrails`.
- [ ] Новый смысл имеет уникальные `experiment_id`, `run_name` и модуль.

После изменения реализации эксперимента выполняйте `Restart Kernel → Run All`.


In [ ]:
from pathlib import Path
import importlib
import sys

from IPython.display import display

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = next(
    candidate for candidate in (CURRENT_DIR, *CURRENT_DIR.parents)
    if (candidate / "README.md").exists() and (candidate / "src").exists()
)
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from ml_project import DataCatalog
import ml_project.config as project_config
import ml_project.baseline_config as baseline_config
import ml_project.experiment_config as experiment_config
import ml_project.modeling as modeling_tools
import ml_project.experiment as experiment_tools

print(f"Корень проекта: {PROJECT_ROOT}")


## 1. Перечитать выбранный эксперимент

Эта ячейка загружает готовый объект `EXPERIMENT` и функции candidate-data /
candidate-models из отдельного модуля. Скрытой сборки конфигурации нет.


In [ ]:
project_config = importlib.reload(project_config)
baseline_config = importlib.reload(baseline_config)
experiment_config = importlib.reload(experiment_config)
modeling_tools = importlib.reload(modeling_tools)
experiment_tools = importlib.reload(experiment_tools)

DATASETS = project_config.DATASETS
FEATURE_GROUPS = project_config.FEATURE_GROUPS
INFERENCE_DATASET = project_config.INFERENCE_DATASET
KEY = project_config.KEY
RAW_DIR = project_config.RAW_DIR
TARGET = project_config.TARGET
TRAIN_DATASET = project_config.TRAIN_DATASET

baseline_settings = baseline_config.BASELINE
experiment_definition = experiment_tools.load_experiment(
    experiment_config.EXPERIMENT_MODULE
)
experiment_settings = experiment_definition.settings

modeling_tools.validate_baseline_settings(baseline_settings)
experiment_tools.validate_settings(experiment_settings)
display(experiment_tools.settings_report(experiment_settings))
print("Модуль:", experiment_definition.module_name)
print("Файл:", experiment_definition.source_path.relative_to(PROJECT_ROOT))
print("SHA-256 кода:", experiment_definition.source_sha256)


## 2. Загрузить неизменяемый data contract

Inference загружается только для проверки схемы и не участвует в оценке.


In [ ]:
catalog = DataCatalog(PROJECT_ROOT, RAW_DIR, DATASETS)
catalog.validate()
train = catalog.load(TRAIN_DATASET)
inference = catalog.load(INFERENCE_DATASET) if INFERENCE_DATASET else None

train_file = catalog.file_report().set_index("dataset").loc[TRAIN_DATASET]
dataset_version = str(train_file["sha256"])
print(f"Train: {train.shape[0]} строк × {train.shape[1]} столбцов")
print("Dataset SHA-256:", dataset_version)


## 3. Автоматически восстановить baseline reference

Reference всегда пересчитывается из `baseline_config.BASELINE` на тех же folds,
а не копируется из старого CSV.


In [ ]:
reference_plan = modeling_tools.resolve_feature_plan(
    train, FEATURE_GROUPS, target=TARGET, key=KEY, settings=baseline_settings,
)
if inference is not None:
    modeling_tools.validate_inference_schema(
        inference,
        reference_plan,
        strict=baseline_settings.require_inference_features,
    )
reference_data = modeling_tools.prepare_training_data(
    train, target=TARGET, plan=reference_plan, settings=baseline_settings,
)
scoring_plan = modeling_tools.resolve_scoring_plan(
    PROJECT_ROOT, baseline_settings,
)
cv, resolved_cv_strategy = modeling_tools.build_cv_splitter(
    baseline_settings, reference_data.y,
)
cv_description = modeling_tools.cv_protocol_description(
    baseline_settings, resolved_cv_strategy,
)
reference_preprocessor = modeling_tools.build_tabular_preprocessor(
    baseline_settings, reference_plan,
)
reference_pipeline = modeling_tools.build_model_pipeline(
    reference_preprocessor,
    modeling_tools.build_simple_estimator(baseline_settings),
)
print("Reference:", experiment_settings.reference_model)
print("Validation:", cv_description)
display(scoring_plan.to_frame())


## 4. Выполнить candidate-data hook

Код находится в выбранном experiment-модуле. Диагностические таблицы можно
вернуть через `ExperimentData.diagnostics`; они отобразятся автоматически.


In [ ]:
candidate_data = experiment_tools.prepare_experiment_candidate(
    experiment_definition,
    train,
    FEATURE_GROUPS,
    baseline_settings,
)
for diagnostic_name, diagnostic in candidate_data.diagnostics.items():
    print(diagnostic_name)
    display(diagnostic)

candidate_plan = modeling_tools.resolve_feature_plan(
    candidate_data.frame,
    candidate_data.feature_groups,
    target=TARGET,
    key=KEY,
    settings=candidate_data.settings,
)
prepared = experiment_tools.prepare_experiment_data(
    reference_data,
    candidate_data.frame,
    target=TARGET,
)
candidate_preprocessor = modeling_tools.build_tabular_preprocessor(
    candidate_data.settings,
    candidate_plan,
)
display(candidate_plan.to_frame())
print("Candidate matrix:", prepared.X.shape)


## 5. Собрать candidate-models из experiment-модуля

Runner проверяет наличие `primary_candidate` и запрещает перезапись reference.


In [ ]:
candidate_models = experiment_tools.build_experiment_candidates(
    experiment_definition,
    candidate_preprocessor,
    candidate_data.settings,
)
models = {
    experiment_settings.reference_model: reference_pipeline,
    **candidate_models,
}
print("Сравниваемые модели:", ", ".join(models))
for model_name, model in candidate_models.items():
    print(model_name)
    display(model)


## 6. Сравнить на одинаковых folds и проверить критерии

Положительный `improvement` всегда означает улучшение, включая minimize-метрики.
Решение остаётся ручным даже при прохождении формальных критериев.


In [ ]:
evaluation = modeling_tools.evaluate_models_cv(
    models,
    prepared,
    cv=cv,
    scoring=scoring_plan,
    settings=baseline_settings,
)
comparison = experiment_tools.comparison_summary(
    evaluation,
    scoring_plan,
    reference_model=experiment_settings.reference_model,
)
criteria = experiment_tools.success_criteria_report(
    evaluation,
    scoring_plan,
    experiment_settings,
)
display(
    comparison[
        [
            "model",
            "metric",
            "direction",
            "mean ± std",
            "reference_mean",
            "improvement",
        ]
    ].round(4)
)
display(criteria.round(4))
print(
    "Формальные критерии:",
    "PASSED" if bool(criteria["passed"].all()) else "FAILED",
)


## 7. Построить графики

PNG сохранятся в `assets/experiments/<EXP-ID>/` при финальном write-action.


In [ ]:
metric_figures = modeling_tools.build_metric_figures(
    evaluation,
    scoring_plan,
)
for metric_key, figure in metric_figures.items():
    print("Метрика:", scoring_plan.labels[metric_key])
    display(figure)


## 8. Сохранить артефакты и синхронизировать отчёты

Metadata получает полный dataset hash, путь и SHA-256 experiment-модуля,
формальные критерии и снимок окружения.


In [ ]:
saved_run = None
write_requested = (
    experiment_settings.save_artifacts
    or experiment_settings.save_final_model
)
if write_requested:
    saved_run = experiment_tools.save_experiment_run(
        PROJECT_ROOT,
        experiment_settings,
        baseline_settings,
        evaluation,
        candidate_plan,
        scoring_plan,
        dataset_version=dataset_version,
        cv_description=cv_description,
        models=models,
        data=prepared,
        metric_figures=metric_figures,
        definition=experiment_definition,
    )
    print("Артефакты:", saved_run.run_dir.relative_to(PROJECT_ROOT))
else:
    print("Запись артефактов отключена.")

if experiment_settings.sync_experiment_note:
    if saved_run is None:
        print("Карточка не обновлена: сначала включите save_artifacts.")
    else:
        updated_note = experiment_tools.sync_experiment_note(
            PROJECT_ROOT,
            experiment_settings,
            evaluation,
            scoring_plan,
            saved_run,
            dataset_version=dataset_version,
            cv_description=cv_description,
            definition=experiment_definition,
        )
        print("Карточка:", experiment_settings.experiment_note, updated_note)

if experiment_settings.sync_docs:
    updated_docs = experiment_tools.sync_experiment_docs(
        PROJECT_ROOT,
        experiment_settings,
        evaluation,
        scoring_plan,
        baseline_settings=baseline_settings,
        dataset_version=dataset_version,
        definition=experiment_definition,
    )
    print("Docs/registry:", updated_docs)

report_issues = modeling_tools.audit_modeling_report(PROJECT_ROOT)
if report_issues:
    print("Проверка целостности отчёта:")
    for issue in report_issues:
        print("-", issue)
else:
    print("Проверка целостности отчёта: OK")


## 9. Решение и следующий эксперимент

### Закрыть текущий эксперимент

1. Объясните результат в созданной карточке.
2. Измените только `EXPERIMENT.decision` в experiment-модуле.
3. Если kernel и evaluation не менялись, повторите ячейки **1** и **8**.
4. При любом изменении данных, признаков или модели выполните полный `Run All`.

### Создать следующий эксперимент

Из корня проекта:

```powershell
.\new-experiment.cmd
```

Launcher сам предложит следующий ID, спросит название, slug, минимальный Δ
и guardrails, затем создаст новый модуль и выберет его в
`experiment_config.py`.
После заполнения `CHANGE ME`, candidate hooks и guardrails запустите этот
notebook сверху вниз. Старые experiment-модули не изменяются и остаются
точными воспроизводимыми спецификациями своих запусков.
